# Train GPT-2 from scratch on your growing corpus

Choose **Runtime → Change runtime type → A100 GPU**. This notebook uses your current `corpus-list.py` and cleaning rules, trains a fresh byte-level tokenizer on training books only, and initializes `GPT2LMHeadModel` with random weights. It does not download pretrained weights or a pretrained tokenizer.

On your computer, run `python3 make_colab_bundle.py`. Open this notebook in Colab; the upload cell will ask for `colab_project.zip`. Books, audits, models, and resumable optimizer checkpoints are saved in Google Drive. Token files live under `/content` and are rebuilt if the runtime is lost. Allow several GB of free Drive storage.

The default schedule is three epochs, with a two-hour training budget per session. Evaluation, checkpoint saving, and export can exceed that budget. Resume to finish the original schedule. Generation quality depends on corpus size and validation behavior; three epochs is a starting experiment.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import io, json, os, subprocess, sys, zipfile

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/GPT2Corpus')
PROJECT = ROOT / 'project'
CORPUS = ROOT / 'corpus'
RUN_NAME = 'gpt2-001'  # Change for a new corpus, tokenizer, architecture, or schedule.
OUTPUT = ROOT / 'runs' / RUN_NAME
MAX_SESSION_SECONDS = 2 * 60 * 60
ALLOW_PARTIAL_CORPUS = False  # True uses accepted books only, after inspecting the audit.
REBUILD_CORPUS = False       # True re-cleans cached books and downloads new list entries.
UPLOAD_UPDATED_PROJECT = False  # True uploads a rebuilt zip after local list/script edits.
PROJECT.mkdir(parents=True, exist_ok=True)


In [ ]:
if UPLOAD_UPDATED_PROJECT or not (PROJECT / 'train_gpt2.py').exists():
    print('Upload colab_project.zip, generated by make_colab_bundle.py.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one project zip.')
    payload = next(iter(uploaded.values()))
    allowed = {'corpus-list.py', 'corpus_registry.py', 'prepare_corpus.py', 'cleaning_rules.json',
               'training_data.py', 'train_gpt2.py', 'training_config.json', 'generate.py',
               'requirements-colab.txt', 'README.md'}
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        if set(archive.namelist()) != allowed:
            raise ValueError('Use the zip produced by make_colab_bundle.py.')
        for name in allowed:
            (PROJECT / name).write_bytes(archive.read(name))
    del uploaded, payload
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Project:', PROJECT)


In [ ]:
# Install before importing ML packages. Keep Colab's CUDA-enabled torch.
# If Colab asks for a runtime restart, restart and rerun from the first cell.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements-colab.txt')], check=True)
subprocess.run(['nvidia-smi'], check=True)


## Prepare and review

The Python list is the only corpus registry. Append more URLs at any time. For volumes or alternate editions, use dictionary entries with a shared `work_id` (see README). After changes, set `REBUILD_CORPUS=True` and choose a new `RUN_NAME`.

Ambiguous cleaning results are quarantined. The next cell prints findings and saves `audit.html` alongside the split files. Review it in Drive and add checksum-bound rules in `cleaning_rules.json`, or explicitly choose the accepted subset with `ALLOW_PARTIAL_CORPUS=True`. No flagged text is included in training.


In [ ]:
import hashlib

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def find_prepared_run():
    pointer = json.loads((CORPUS / 'latest.json').read_text())
    return CORPUS / pointer.get('relative_run', 'runs/' + Path(pointer['run']).name)

latest = CORPUS / 'latest.json'
if latest.exists() and not REBUILD_CORPUS:
    prepared = find_prepared_run()
    previous = json.loads((prepared / 'report.json').read_text())
    if (previous['sources_sha256'] != digest(PROJECT / 'corpus-list.py') or
        previous['rules_sha256'] != digest(PROJECT / 'cleaning_rules.json')):
        raise RuntimeError('List/rules changed. Set REBUILD_CORPUS=True and choose a new RUN_NAME.')
else:
    command = [sys.executable, str(PROJECT / 'prepare_corpus.py'),
               '--output', str(CORPUS), '--cache-raw']
    previous_pointer = latest.read_bytes() if latest.exists() else None
    completed = subprocess.run(command)
    if completed.returncode not in (0, 1) or not latest.exists():
        raise RuntimeError('Preparation failed before producing a run; inspect its output.')
    if latest.read_bytes() == previous_pointer:
        raise RuntimeError('Preparation produced no new run; refusing to use stale outputs.')
    prepared = find_prepared_run()
report = json.loads((prepared / 'report.json').read_text())
print(json.dumps(report['counts'], indent=2))
print('Audit:', prepared / 'audit.html')
manifest = [json.loads(line) for line in (prepared / 'manifest.jsonl').read_text().splitlines()]
for row in manifest:
    if row['status'] in ('needs_review', 'failed'):
        print(row['id'], row['status'], row.get('problems', row.get('error')))
if not report['complete'] and not ALLOW_PARTIAL_CORPUS:
    raise RuntimeError('Review audit.html and cleaning_rules.json, or explicitly set ALLOW_PARTIAL_CORPUS=True.')


## Train or resume

Adjust `training_config.json` **before starting a new run**. Defaults: 12 layers, 12 heads, 768 hidden width, 1,024-token context, up to 32,768 tokenizer entries, microbatch 8, accumulation 8. Uses BF16, TF32, PyTorch SDPA, fused AdamW, gradient checkpointing, warmup and cosine decay.

The program prints actual book/token counts, parameter count, optimizer steps, and measured throughput. It validates every book against the cleaning manifest and requires nonempty train and validation splits. Best checkpoint selection uses validation loss; test loss is computed only after the training schedule completes.

Rerun this cell in a later session to resume. It restores weights, optimizer, scheduler, RNG state, and the position in the shuffled epoch. Resume requires the same corpus, config, code, tokenizer and package versions. Colab GPU kernels may still be nondeterministic. A CUDA out-of-memory error requires a smaller batch size in a **new** run; increase accumulation to preserve effective batch size.


In [ ]:
command = [sys.executable, str(PROJECT / 'train_gpt2.py'),
           '--corpus', str(prepared), '--output', str(OUTPUT),
           '--work-dir', '/content/gpt2-token-cache',
           '--config', str(PROJECT / 'training_config.json'),
           '--max-seconds', str(MAX_SESSION_SECONDS)]
if (OUTPUT / 'run.json').exists():
    command.append('--resume')
if ALLOW_PARTIAL_CORPUS:
    command.append('--allow-partial-corpus')
subprocess.run(command, check=True)
result = json.loads((OUTPUT / 'result.json').read_text())
print(json.dumps(result, indent=2))


## Generate

`best_model` is a self-contained export of the best evaluated checkpoint. A run stopped at the session budget is usable for sampling and can still be resumed. Keep resumable checkpoints on Drive; the inference export alone does not include optimizer state.


In [ ]:
PROMPT = 'The evening was quiet, and'
subprocess.run([sys.executable, str(PROJECT / 'generate.py'),
                '--model', str(OUTPUT / 'best_model'), '--prompt', PROMPT,
                '--max-new-tokens', '200', '--temperature', '0.8', '--top-p', '0.95'], check=True)


References: [GPT-2 configuration and causal language-model head](https://huggingface.co/docs/transformers/model_doc/gpt2), [byte BPE trainer](https://huggingface.co/docs/tokenizers/api/trainers), [PyTorch automatic mixed precision](https://docs.pytorch.org/docs/stable/amp.html), [Colab runtime and Drive limitations](https://research.google.com/colaboratory/faq.html). GPU availability and session duration vary; the script records measured throughput rather than promising a completion time.
